# Binoculars — standalone (Kaggle)

**Jeden notebook = cały kod.** Bez zewnętrznych plików / datasetów.

Każda generacja i każda ewaluacja idzie w **osobnym procesie** → VRAM jest czyszczona między Gemma a Falconem (unikamy OOM).

Wymagania: **GPU**, Internet, secret `HF_TOKEN`.

Na koniec: `binoculars_summary.csv` + zip w Output.

In [ ]:
# transformers 5.x OOMuuje przy 4-bit (ładuje fp16 przed kwantyzacją) → pin 4.57
!pip install -q "transformers==4.57.5" "accelerate>=0.33.0,<1.0" \
                "bitsandbytes>=0.43.1" "sentencepiece" "protobuf" \
                "numpy" "tqdm" "huggingface-hub"

import os
os.environ['HF_DEACTIVATE_ASYNC_LOAD'] = '1'  # dodatkowy bezpiecznik na v5

import bitsandbytes  # noqa: F401
import transformers
print('transformers', transformers.__version__)
print('OK')

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ['HF_DEACTIVATE_ASYNC_LOAD'] = '1'

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF OK')

import torch
print('torch.cuda.is_available():', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'Brak GPU w tej sesji!\n'
        '1) Prawy panel → Settings → Accelerator → GPU T4 / P100\n'
        '2) Restart session\n'
        '3) Przy Save Version też ustaw Accelerator=GPU (domyślnie bywa CPU)'
    )
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi -L

In [ ]:
# === KONFIGURACJA ===
MODELS = ['qwen', 'llama', 'gemma']
THRESHOLDS = [0.0, 0.01, 0.05, 0.1]

# Po OOM / restarcie — odpal tylko brakujące, np.:
# MODELS = ['gemma']
# THRESHOLDS = [0.0, 0.01, 0.05, 0.1]

TOP_N = 16
MAX_NEW_TOKENS = 512
SEED = 1234
SKIP_IF_DONE = True
CONTINUE_ON_ERROR = True

from pathlib import Path

WORK = Path('/kaggle/working')
RUNS_DIR = WORK / 'binoculars_runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
WORKER = WORK / 'bino_worker.py'
print('RUNS_DIR:', RUNS_DIR)

In [ ]:
# === ZAPIS WORKERA (odpala się w osobnym procesie = czyste GPU) ===
WORKER.write_text(r'''#!/usr/bin/env python3
"""Standalone binoculars worker — gen | eval."""
from __future__ import annotations

import argparse
import gc
import json
import math
import os
import sys
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    LogitsProcessor,
    LogitsProcessorList,
)

MODEL_IDS = {
    "qwen": "Qwen/Qwen2.5-7B-Instruct",
    "llama": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "gemma": "google/gemma-2-9b-it",
}
PROMPTS = [
    "Write a short paragraph explaining how binary search works.",
    "Describe the difference between a stack and a queue in computer science.",
    "Explain what a hash table is and when you would use one.",
    "Write a concise summary of how gradient descent optimizes neural networks.",
]
BINO_OBSERVER = "tiiuae/falcon-7b"
BINO_PERFORMER = "tiiuae/falcon-7b-instruct"
BINO_FPR_THRESHOLD = 0.8536432310785527
MAX_GPU_GB = 7.5


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def vram(tag: str = "") -> None:
    if not torch.cuda.is_available():
        return
    a = torch.cuda.memory_allocated() / (1024 ** 3)
    r = torch.cuda.memory_reserved() / (1024 ** 3)
    print(f"VRAM {tag}: alloc={a:.2f}GB reserved={r:.2f}GB", flush=True)


def hf_kwargs() -> dict:
    kw = {"trust_remote_code": False}
    tok = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
    if tok:
        kw["token"] = tok
    return kw


def quant_4bit() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def quant_8bit() -> BitsAndBytesConfig:
    return BitsAndBytesConfig(load_in_8bit=True)


def resolve_model(key: str) -> tuple[str, str]:
    if key in MODEL_IDS:
        return key, MODEL_IDS[key]
    return key, key


def build_prompt(tokenizer, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer clearly and concisely."},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return user_prompt
    return user_prompt


@dataclass
class StegoCapacityStats:
    survivor_counts: list[int] = field(default_factory=list)
    stego_applied_steps: int = 0
    natural_fallback_steps: int = 0

    def record(self, survivor_count: int, stego_applied: bool) -> None:
        self.survivor_counts.append(int(survivor_count))
        if stego_applied:
            self.stego_applied_steps += 1
        else:
            self.natural_fallback_steps += 1

    @property
    def total_steps(self) -> int:
        return len(self.survivor_counts)

    @property
    def avg_pool_size(self) -> float:
        return sum(self.survivor_counts) / len(self.survivor_counts) if self.survivor_counts else 0.0

    @property
    def avg_pool_size_stego_only(self) -> float:
        pools = [c for c in self.survivor_counts if c >= 2]
        return sum(pools) / len(pools) if pools else 0.0

    @property
    def avg_bits_per_token(self) -> float:
        if not self.survivor_counts:
            return 0.0
        bits = [math.log2(c) if c >= 2 else 0.0 for c in self.survivor_counts]
        return sum(bits) / len(bits)

    @property
    def embedding_rate(self) -> float:
        return self.avg_bits_per_token

    @property
    def stego_activation_ratio(self) -> float:
        return self.stego_applied_steps / self.total_steps if self.total_steps else 0.0

    def to_dict(self) -> dict:
        return {
            "total_steps": self.total_steps,
            "stego_applied_steps": self.stego_applied_steps,
            "natural_fallback_steps": self.natural_fallback_steps,
            "stego_activation_ratio": self.stego_activation_ratio,
            "avg_pool_size": self.avg_pool_size,
            "avg_pool_size_stego_only": self.avg_pool_size_stego_only,
            "avg_bits_per_token": self.avg_bits_per_token,
            "embedding_rate": self.embedding_rate,
        }


class DummyStegoLogitsProcessor(LogitsProcessor):
    def __init__(self, top_n: int, threshold: float, seed: int | None = None):
        self.top_n = int(top_n)
        self.threshold = float(threshold)
        self.seed = seed
        self.capacity_stats = StegoCapacityStats()
        self._generator = None
        self._generator_device = None

    def _get_generator(self, device):
        if self.seed is None:
            return None
        if self._generator is None or self._generator_device != device:
            self._generator = torch.Generator(device=device)
            self._generator.manual_seed(self.seed)
            self._generator_device = device
        return self._generator

    def __call__(self, input_ids, scores):
        probs = torch.softmax(scores, dim=-1)
        k = min(self.top_n, probs.shape[-1])
        top_probs, top_indices = torch.topk(probs, k=k, dim=-1)
        survivor_mask = top_probs >= self.threshold
        survivor_count = survivor_mask.sum(dim=-1)
        apply_random = survivor_count >= 2
        for row_idx in range(scores.shape[0]):
            self.capacity_stats.record(
                int(survivor_count[row_idx].item()), bool(apply_random[row_idx].item())
            )
        if not torch.any(apply_random):
            return scores
        random_scores = torch.rand(
            top_probs.shape, device=scores.device, generator=self._get_generator(scores.device)
        )
        random_scores = random_scores.masked_fill(~survivor_mask, -1.0)
        chosen_rank = random_scores.argmax(dim=-1, keepdim=True)
        chosen_token = top_indices.gather(dim=-1, index=chosen_rank)
        forced = torch.full_like(scores, float("-inf"))
        forced.scatter_(dim=-1, index=chosen_token, value=float("inf"))
        return torch.where(apply_random.unsqueeze(-1), forced, scores)


def require_cuda() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "Brak GPU! W Kaggle: Settings → Accelerator → GPU T4/P100, "
            "potem Restart session. Save Version też musi mieć GPU włączone."
        )
    print(f"CUDA OK: {torch.cuda.get_device_name(0)}", flush=True)


def load_generator(model_id: str):
    require_cuda()
    print(f"Loading generator: {model_id}", flush=True)
    print(f"transformers={transformers.__version__} HF_DEACTIVATE_ASYNC_LOAD={os.getenv('HF_DEACTIVATE_ASYNC_LOAD')}", flush=True)
    free_gpu()
    vram("before-gen-load")
    tok = AutoTokenizer.from_pretrained(model_id, **hf_kwargs())
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    # 4-bit: na T4 16GB Llama/Gemma muszą wejść skwantyzowane (~5-7GB), nie fp16 (~14GB+)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_4bit(),
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        **hf_kwargs(),
    )
    model.eval()
    vram("after-gen-load")
    used = torch.cuda.memory_allocated() / (1024 ** 3)
    if used > 10.0:
        raise RuntimeError(
            f"Model zajął {used:.1f} GB — kwantyzacja 4-bit nie zadziałała. "
            "Zainstaluj transformers==4.57.5 i ustaw HF_DEACTIVATE_ASYNC_LOAD=1."
        )
    return model, tok


@torch.no_grad()
def generate_raw(model, tokenizer, prompt_text, *, max_new_tokens, threshold, top_n, seed):
    enc = tokenizer(prompt_text, return_tensors="pt")
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)
    kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    capacity = StegoCapacityStats()
    if threshold > 0.0:
        proc = DummyStegoLogitsProcessor(top_n=top_n, threshold=threshold, seed=seed)
        kwargs["logits_processor"] = LogitsProcessorList([proc])
        out = model.generate(**kwargs)
        capacity = proc.capacity_stats
    else:
        out = model.generate(**kwargs)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0, input_ids.shape[1] :], skip_special_tokens=True).strip()
    free_gpu()
    return full, completion, capacity


def cmd_gen(args) -> int:
    model_key, model_id = resolve_model(args.model)
    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    raw_path = run_dir / "raw_responses.jsonl"
    if raw_path.exists():
        raw_path.unlink()

    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    model, tokenizer = load_generator(model_id)
    records = []
    try:
        for idx, user_prompt in enumerate(tqdm(PROMPTS, desc=f"{model_key} th={args.threshold}")):
            prompt_text = build_prompt(tokenizer, user_prompt)
            base_full, base_comp, _ = generate_raw(
                model, tokenizer, prompt_text,
                max_new_tokens=args.max_new_tokens, threshold=0.0,
                top_n=args.top_n, seed=args.seed,
            )
            stego_full, stego_comp, cap = generate_raw(
                model, tokenizer, prompt_text,
                max_new_tokens=args.max_new_tokens, threshold=args.threshold,
                top_n=args.top_n, seed=args.seed,
            )
            rec = {
                "sample_id": f"binoculars/{idx}",
                "prompt_index": idx,
                "user_prompt": user_prompt,
                "prompt_text": prompt_text,
                "generated_at": utc_now(),
                "baseline": {"raw_full_decoded": base_full, "raw_completion": base_comp},
                "stego": {
                    "raw_full_decoded": stego_full,
                    "raw_completion": stego_comp,
                    "capacity": cap.to_dict(),
                },
            }
            records.append(rec)
            with raw_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"  saved binoculars/{idx}", flush=True)
    finally:
        del model, tokenizer
        free_gpu()
        vram("after-gen-release")

    (run_dir / "manifest.json").write_text(
        json.dumps(
            {
                "phase": "generation",
                "status": "completed",
                "test": "binoculars",
                "model_key": model_key,
                "model_id": model_id,
                "threshold": args.threshold,
                "top_n": args.top_n,
                "max_new_tokens": args.max_new_tokens,
                "seed": args.seed,
                "platform": "kaggle",
                "completed_count": len(records),
                "total_tasks": len(PROMPTS),
                "updated_at": utc_now(),
                "completed_at": utc_now(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Generation complete: {run_dir}", flush=True)
    return 0


_CE = torch.nn.CrossEntropyLoss(reduction="none")
_SM = torch.nn.Softmax(dim=-1)


def _gpu_gb() -> float:
    return torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0


def _device(model):
    return next(model.parameters()).device


def _bino_ppl(encoding, logits):
    shifted_logits = logits[..., :-1, :].contiguous()
    shifted_labels = encoding["input_ids"][..., 1:].contiguous()
    shifted_mask = encoding["attention_mask"][..., 1:].contiguous()
    losses = _CE(shifted_logits.transpose(1, 2), shifted_labels)
    return ((losses * shifted_mask).sum(1) / shifted_mask.sum(1)).cpu().float().numpy()


def _bino_xppl(obs_logits, perf_logits, encoding, pad_id):
    vocab = obs_logits.shape[-1]
    T = perf_logits.shape[-2]
    p = _SM(obs_logits).view(-1, vocab)
    q = perf_logits.view(-1, vocab)
    ce = _CE(input=q, target=p).view(-1, T)
    mask = (encoding["input_ids"] != pad_id).type(torch.uint8)
    return ((ce * mask).sum(1) / mask.sum(1)).cpu().float().numpy()


class BinocularsScorer:
    def __init__(self):
        self.threshold = BINO_FPR_THRESHOLD
        self.tokenizer = AutoTokenizer.from_pretrained(BINO_OBSERVER, **hf_kwargs())
        if not self.tokenizer.pad_token:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _load(self, model_id: str):
        base = dict(
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            attn_implementation="eager",
            **hf_kwargs(),
        )
        last = None
        for label, qcfg in [("4bit", quant_4bit()), ("8bit", quant_8bit())]:
            free_gpu()
            try:
                print(f"  Loading {model_id} ({label})...", flush=True)
                m = AutoModelForCausalLM.from_pretrained(
                    model_id, quantization_config=qcfg, **base
                )
                used = _gpu_gb()
                if used > MAX_GPU_GB:
                    print(f"  {label} too heavy ({used:.1f} GB) — next", flush=True)
                    del m
                    free_gpu()
                    continue
                print(f"  Loaded {label}, VRAM ~{used:.1f} GB", flush=True)
                return m
            except Exception as e:
                last = e
                print(f"  {label} failed: {e}", flush=True)
                free_gpu()
        raise RuntimeError(f"Cannot load {model_id}: {last}")

    @torch.inference_mode()
    def score(self, text: str) -> float:
        if not text.strip():
            raise ValueError("empty text")
        enc = self.tokenizer(
            text, return_tensors="pt", truncation=True, max_length=512, return_token_type_ids=False
        )
        enc_cpu = {k: v.cpu() for k, v in enc.items()}

        obs = self._load(BINO_OBSERVER)
        enc_g = {k: v.to(_device(obs)) for k, v in enc_cpu.items()}
        obs_logits = obs(**enc_g).logits.detach().cpu()
        del obs, enc_g
        free_gpu()

        perf = self._load(BINO_PERFORMER)
        enc_g = {k: v.to(_device(perf)) for k, v in enc_cpu.items()}
        perf_logits = perf(**enc_g).logits.detach().cpu()
        del perf, enc_g
        free_gpu()

        ppl = _bino_ppl(enc_cpu, perf_logits)
        xppl = _bino_xppl(obs_logits, perf_logits, enc_cpu, self.tokenizer.pad_token_id)
        return float((ppl / xppl)[0])

    def label(self, score: float) -> str:
        return "Most likely AI-generated" if score < self.threshold else "Most likely human-generated"


def merge_capacity(caps: list[dict]) -> dict:
    if not caps:
        return {}
    total = sum(int(c.get("total_steps", 0)) for c in caps)
    stego = sum(int(c.get("stego_applied_steps", 0)) for c in caps)
    natural = sum(int(c.get("natural_fallback_steps", 0)) for c in caps)
    if total == 0:
        return {
            "total_steps": 0, "stego_applied_steps": 0, "natural_fallback_steps": 0,
            "stego_activation_ratio": 0.0, "avg_pool_size": 0.0, "avg_pool_size_stego_only": 0.0,
            "avg_bits_per_token": 0.0, "embedding_rate": 0.0,
        }
    w_pool = sum(float(c.get("avg_pool_size", 0)) * int(c.get("total_steps", 0)) for c in caps)
    w_pool_s = sum(
        float(c.get("avg_pool_size_stego_only", 0)) * int(c.get("stego_applied_steps", 0)) for c in caps
    )
    w_bpt = sum(float(c.get("avg_bits_per_token", 0)) * int(c.get("total_steps", 0)) for c in caps)
    return {
        "total_steps": total,
        "stego_applied_steps": stego,
        "natural_fallback_steps": natural,
        "stego_activation_ratio": stego / total,
        "avg_pool_size": w_pool / total,
        "avg_pool_size_stego_only": (w_pool_s / stego) if stego else 0.0,
        "avg_bits_per_token": w_bpt / total,
        "embedding_rate": w_bpt / total,
    }


def cmd_eval(args) -> int:
    require_cuda()
    run_dir = Path(args.run_dir)
    records = []
    with (run_dir / "raw_responses.jsonl").open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    free_gpu()
    vram("before-eval")
    scorer = BinocularsScorer()
    samples = []
    base_scores, stego_scores = [], []

    for rec in tqdm(records, desc="binoculars scoring"):
        base_text = rec["baseline"]["raw_completion"].strip()
        stego_text = rec["stego"]["raw_completion"].strip()
        b = scorer.score(base_text)
        s = scorer.score(stego_text)
        base_scores.append(b)
        stego_scores.append(s)
        samples.append(
            {
                "sample_id": rec["sample_id"],
                "user_prompt": rec["user_prompt"],
                "baseline_generated": base_text,
                "stego_generated": stego_text,
                "baseline_binoculars_score": b,
                "stego_binoculars_score": s,
                "baseline_prediction": scorer.label(b),
                "stego_prediction": scorer.label(s),
                "capacity": rec["stego"].get("capacity", {}),
            }
        )

    mean_b = sum(base_scores) / len(base_scores)
    mean_s = sum(stego_scores) / len(stego_scores)
    ai_b = sum(x < scorer.threshold for x in base_scores) / len(base_scores)
    ai_s = sum(x < scorer.threshold for x in stego_scores) / len(stego_scores)
    cap = merge_capacity([r.get("capacity", {}) for r in samples])
    results = {
        "test": "binoculars",
        "evaluated_at": utc_now(),
        "binoculars_score": mean_s,
        "baseline_binoculars_score": mean_b,
        "binoculars_score_delta": mean_s - mean_b,
        "ai_detection_rate": ai_s,
        "baseline_ai_detection_rate": ai_b,
        "binoculars_threshold": scorer.threshold,
        "capacity": cap,
        "samples": samples,
    }
    eval_dir = run_dir / "evaluation"
    eval_dir.mkdir(parents=True, exist_ok=True)
    (eval_dir / "binoculars_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
    del scorer
    free_gpu()
    vram("after-eval")
    print(
        f"=== Binoculars: baseline={mean_b:.3f} stego={mean_s:.3f} AI%={ai_s:.1%} ===",
        flush=True,
    )
    return 0


def main() -> int:
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)

    g = sub.add_parser("gen")
    g.add_argument("--model", required=True)
    g.add_argument("--threshold", type=float, required=True)
    g.add_argument("--run-dir", required=True)
    g.add_argument("--top-n", type=int, default=16)
    g.add_argument("--max-new-tokens", type=int, default=512)
    g.add_argument("--seed", type=int, default=1234)

    e = sub.add_parser("eval")
    e.add_argument("--run-dir", required=True)

    args = p.parse_args()
    if args.cmd == "gen":
        return cmd_gen(args)
    return cmd_eval(args)


if __name__ == "__main__":
    raise SystemExit(main())
''')
print('Worker zapisany:', WORKER)

In [ ]:
# === GŁÓWNA PĘTLA (subprocess = czyste GPU między runami) ===
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Brak GPU — włącz Accelerator=GPU i Restart session przed pętlą.')

jobs = [(m, th) for m in MODELS for th in THRESHOLDS]
print(f'Plan: {len(jobs)} runów (każdy gen+eval = 2 procesy)')
print('GPU:', torch.cuda.get_device_name(0))


def find_existing(model_key: str, threshold: float) -> Path | None:
    best, best_t = None, ''
    for man in RUNS_DIR.glob('*/manifest.json'):
        try:
            m = json.loads(man.read_text())
        except Exception:
            continue
        if m.get('model_key') != model_key:
            continue
        if abs(float(m.get('threshold', -1)) - float(threshold)) > 1e-9:
            continue
        if m.get('status') != 'completed':
            continue
        t = m.get('updated_at', '')
        if t >= best_t:
            best_t, best = t, man.parent
    return best


def has_eval(run_dir: Path) -> bool:
    return (run_dir / 'evaluation' / 'binoculars_results.json').is_file()


def run_slug(model_key: str, threshold: float) -> str:
    ts = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    th = str(threshold).replace('.', '_')
    return f'{ts}_{model_key}_binoculars_th{th}'


def run_live(cmd: list[str]) -> None:
    print('CMD:', ' '.join(cmd), flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    env['HF_DEACTIVATE_ASYNC_LOAD'] = '1'
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    assert p.stdout is not None
    for line in p.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    rc = p.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


ok, skipped, failed = [], [], []

for i, (model_key, threshold) in enumerate(jobs, start=1):
    label = f'{model_key} | th={threshold}'
    print('\n' + '=' * 60)
    print(f'[{i}/{len(jobs)}] {label}')
    print('=' * 60)

    run_dir = find_existing(model_key, threshold)

    if SKIP_IF_DONE and run_dir is not None and has_eval(run_dir):
        print(f'Pomijam (gotowe): {run_dir.name}')
        skipped.append(label)
        continue

    try:
        if run_dir is None:
            run_dir = RUNS_DIR / run_slug(model_key, threshold)
            print(f'[1/2] Generacja (osobny proces) → {run_dir.name}')
            run_live([
                sys.executable, '-u', str(WORKER), 'gen',
                '--model', model_key,
                '--threshold', str(threshold),
                '--run-dir', str(run_dir),
                '--top-n', str(TOP_N),
                '--max-new-tokens', str(MAX_NEW_TOKENS),
                '--seed', str(SEED),
            ])
        else:
            print(f'[1/2] RAW OK: {run_dir.name}')

        print('[2/2] Binoculars eval (osobny proces)...')
        run_live([
            sys.executable, '-u', str(WORKER), 'eval',
            '--run-dir', str(run_dir),
        ])

        if not has_eval(run_dir):
            raise RuntimeError('Brak binoculars_results.json')

        data = json.loads((run_dir / 'evaluation' / 'binoculars_results.json').read_text())
        ok.append(label)
        print(
            f"OK: baseline={data['baseline_binoculars_score']:.3f} "
            f"stego={data['binoculars_score']:.3f} "
            f"AI={data['ai_detection_rate']:.1%}"
        )
    except Exception as e:
        print(f'BŁĄD: {e}')
        failed.append((label, str(e)))
        if not CONTINUE_ON_ERROR:
            raise

print('\n' + '#' * 40)
print(f'OK={len(ok)} skip={len(skipped)} fail={len(failed)}')
if failed:
    for label, err in failed:
        print(f'  - {label}: {err}')

In [ ]:
# === CSV + ZIP ===
import csv
import json
import shutil
from datetime import datetime
from pathlib import Path

rows = []
for man in sorted(RUNS_DIR.glob('*/manifest.json')):
    m = json.loads(man.read_text())
    ev = man.parent / 'evaluation' / 'binoculars_results.json'
    if not ev.is_file():
        print(f"{m.get('model_key')} th={m.get('threshold')} — brak eval")
        continue
    d = json.loads(ev.read_text())
    rows.append({
        'model': m.get('model_key'),
        'threshold': m.get('threshold'),
        'run': man.parent.name,
        'baseline': d['baseline_binoculars_score'],
        'stego': d['binoculars_score'],
        'delta': d['binoculars_score_delta'],
        'ai_rate': d['ai_detection_rate'],
    })

rows.sort(key=lambda r: (r['model'], float(r['threshold'])))
csv_path = WORK / 'binoculars_summary.csv'
cols = ['model', 'threshold', 'run', 'baseline', 'stego', 'delta', 'ai_rate']
with csv_path.open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(rows)

print(f'CSV: {csv_path} ({len(rows)} wierszy)')
print(f"{'model':<6} {'th':<5} {'baseline':>9} {'stego':>9} {'delta':>9} {'AI%':>7}")
print('-' * 55)
for r in rows:
    print(f"{r['model']:<6} {r['threshold']:<5} {r['baseline']:9.3f} {r['stego']:9.3f} {r['delta']:+9.3f} {100*r['ai_rate']:6.1f}%")

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
archive = shutil.make_archive(str(WORK / f'binoculars_runs_{ts}'), 'zip', RUNS_DIR)
print(f'\nZIP: {archive}')
print('Pobierz ZIP + CSV z Output ZANIM zamkniesz sesję.')